## Basic RAG with Ollama + ChromaDB

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline
using local Ollama models and ChromaDB as the vector store.


In [1]:
import os
import sys
import subprocess
import time

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text-v2-moe:latest
    !ollama pull llama3.1:8b-instruct-q4_K_M

In [2]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import time

import pandas as pd
from datasets import load_dataset
from langchain_core.runnables import RunnableConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import build_rag_chain, TimingCallbackHandler
from reliablerag.env import load_secrets
from reliablerag.evaluation import evaluate
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import get_hyde_retriever, get_hybrid_retriever, get_hybrid_reranked_retriever, get_or_build_vector_store, get_reranked_retriever, get_reranker, get_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [3]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
GENERATOR_MODEL    = os.environ["GENERATOR_MODEL"]
JUDGE_MODEL        = os.environ["JUDGE_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"Judge model     : {JUDGE_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

Provider        : ollama
Embedding model : nomic-embed-text-v2-moe:latest
Generator model : gemma4:12b-it-q4_K_M
Judge model     : llama3.1:8b-instruct-q4_K_M
Chroma dir      : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [4]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)

# Generator: large model used to write the final answer.
llm = create_llm(PROVIDER, GENERATOR_MODEL)

# Judge: smaller model used by the TRACe eval. Deterministic decoding so
# repeat evals on the same inputs don't drift. Bumped to a smaller open-source
# model (Llama 3.1 8B) — gemma 12B as judge was making each eval call ~3-4min.
judge_llm = create_llm(PROVIDER, JUDGE_MODEL, temperature=0)

# Cross-encoder reranker. Built once outside the per-sample loop — the
# model weights (~280MB for bge-reranker-base) are downloaded on first use.
reranker = get_reranker()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### 2. Load CUAD Samples from RAGBench

Each sample contains a legal contract (`documents`), a `question`, a reference `response` generated
by Claude 3 Haiku, and pre-computed **TRACe labels** annotated by GPT-4:
`adherence`, `context_relevance`, `utilization`, `completeness`.

In [5]:
N_SAMPLES = 20  # increase to evaluate more samples

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))


def fmt(v):
    return f"{v:.3f}" if v is not None else "N/A"


print(f"Loaded {len(samples)} CUAD samples")

s = samples[0]
print(f"\nQuestion         : {s['question']}")
print(f"Doc length       : {len(s['documents'][0])} chars")
print(f"Adherence score  : {s['adherence_score']}")
print(f"Relevance score  : {fmt(s['relevance_score'])}")
print(f"Utilization score: {fmt(s['utilization_score'])}")
print(f"Completeness     : {fmt(s['completeness_score'])}")
print(f"\nAlso available   : ragas_faithfulness={fmt(s['ragas_faithfulness'])}, "
      f"trulens_groundedness={fmt(s['trulens_groundedness'])}")

Loaded 20 CUAD samples

Question         : Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Doc length       : 122054 chars
Adherence score  : True
Relevance score  : 0.000
Utilization score: 0.000
Completeness     : 1.000

Also available   : ragas_faithfulness=N/A, trulens_groundedness=N/A


In [9]:
df_preview = dataset.to_pandas()
# df_preview.head(10)

### 3. Run RAG on Each Sample

CUAD contracts are up to 11k tokens each, so we chunk each document before indexing.
We build a fresh ephemeral vector store per sample (CUAD has 1 doc per question, so no cross-contamination).

In [8]:
# Step H — Hybrid retrieval (BM25 + dense, RRF fusion).
# Hypothesis: BM25 recovers exact-term clause misses that dense retrieval drops.
# Baseline (Step F): (0.173 / 0.097 / 0.564 / 55%) — completeness is the gap vs ref 0.717.
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 20

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results = []

for i, sample in enumerate(samples):
    question   = sample["question"]
    raw_doc    = sample["documents"][0]  # CUAD: always 1 doc per sample
    coll_name  = f"cuad_{i}_{COLLECTION_TAG}"

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )

    t0 = time.perf_counter()
    vector_store, cached = get_or_build_vector_store(
        chunks, embeddings,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    elapsed = time.perf_counter() - t0
    print(f"[timing] vector store : {elapsed:.3f}s  ({'cache hit' if cached else f'{len(chunks)} chunks embedded'})")

    retriever = get_hybrid_retriever(vector_store, chunks, top_k=TOP_K)

    t0 = time.perf_counter()
    retrieved_chunks = retriever.invoke(question)
    print(f"[timing] retrieve (top-{TOP_K}) : {time.perf_counter() - t0:.3f}s")

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  → {our_response[:100]}...")

    results.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

print(f"\nDone — {len(results)} samples processed.")


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.750s  (291 chunks embedded)
[timing] retrieve (top-20) : 0.166s
[timing] retriever: 0.129s
[timing] retriever: 0.001s
[timing] llm      : 34.051s
  → I do not have enough information in the provided context to determine if a party is required to depo...

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.705s  (47 chunks embedded)
[timing] retrieve (top-20) : 0.138s
[timing] retriever: 0.121s
[timing] retriever: 0.000s
[timing] llm      : 33.599s
  → Based on the provided context, there is no mention of a license being granted by one party to its co...

[3/20] The date when the contract is effective ...
[timing] vector store : 2.075s  (168 chunks embedded)
[timing] retrieve (top-20) : 0.141s
[timing] retriever: 0.117s
[timing] retriever: 0.000s
[timing] llm      : 36.272s
  → The Agreement becomes 

In [12]:
# Step I — Tune RRF weights (bm25_weight=0.3, dense_weight=0.7).
# Step H (equal-weight hybrid, both retrievers weight=1.0) beat baseline completeness (0.564 → 0.590) but dropped adherence (55% → 30%).
# Hypothesis: lower BM25 weight keeps the coverage benefit while reducing noise.
# Baseline (Step F / dense only): (0.173 / 0.097 / 0.564 / 55%)
# Step H (equal weights 0.5/0.5):  (0.112 / 0.086 / 0.590 / 30%)
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 20
BM25_WEIGHT   = 0.3   # ← tunable; dense gets 1 - BM25_WEIGHT = 0.7

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_i = []

for i, sample in enumerate(samples):
    question   = sample["question"]
    raw_doc    = sample["documents"][0]
    coll_name  = f"cuad_{i}_{COLLECTION_TAG}"

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )

    t0 = time.perf_counter()
    vector_store, cached = get_or_build_vector_store(
        chunks, embeddings,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    elapsed = time.perf_counter() - t0
    print(f"[timing] vector store : {elapsed:.3f}s  ({'cache hit' if cached else f'{len(chunks)} chunks embedded'})")

    retriever = get_hybrid_retriever(vector_store, chunks, top_k=TOP_K, bm25_weight=BM25_WEIGHT)

    t0 = time.perf_counter()
    retrieved_chunks = retriever.invoke(question)
    print(f"[timing] retrieve (top-{TOP_K}) : {time.perf_counter() - t0:.3f}s")

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  → {our_response[:100]}...")

    results_i.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

print(f"\nDone — {len(results_i)} samples processed.")


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.564s  (291 chunks embedded)
[timing] retrieve (top-20) : 0.137s
[timing] retriever: 0.115s
[timing] retriever: 0.001s
[timing] llm      : 29.294s
  → The provided text does not contain any mention of "source code" or "escrow" arrangements. Therefore,...

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.605s  (47 chunks embedded)
[timing] retrieve (top-20) : 0.138s
[timing] retriever: 0.115s
[timing] retriever: 0.001s
[timing] llm      : 26.673s
  → Based on the provided text, there is no mention of a license granted by one party to its counterpart...

[3/20] The date when the contract is effective ...
[timing] vector store : 2.005s  (168 chunks embedded)
[timing] retrieve (top-20) : 0.140s
[timing] retriever: 0.121s
[timing] retriever: 0.001s
[timing] llm      : 24.474s
  → The agreement becomes 

In [13]:
# Step I — TRACe evaluation for results_i (bm25_weight=0.3)
JUDGE_N_RUNS = 3

print(f"Evaluating Step I results (bm25_weight={BM25_WEIGHT}, n_runs={JUDGE_N_RUNS})...\n")
for r in results_i:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_parsed_llm_response"]            = scores.parsed_llm_response

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

avg_rel  = sum(r["our_relevance"]    for r in results_i) / len(results_i)
avg_util = sum(r["our_utilization"]  for r in results_i) / len(results_i)
avg_comp = sum(r["our_completeness"] for r in results_i) / len(results_i)
adh_rate = sum(r["our_adherence"]    for r in results_i) / len(results_i)

print(f"Step I  (bm25={BM25_WEIGHT}) — Rel {avg_rel:.3f} / Util {avg_util:.3f} / Comp {avg_comp:.3f} / Adh {adh_rate:.0%}")
print(f"Step H  (bm25=0.5)          — Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

Evaluating Step I results (bm25_weight=0.3, n_runs=3)...

[FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence claims that 
  Relevance   : 0.009 — The relevant information for answering this question can be found in Articles III, VIII, a
  Utilization : 0.004
  Completeness: 0.429

[FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence of the respo
  Relevance   : 0.074 — The relevant information for answering this question can be found in document 1, specifica
  Utilization : 0.074
  Completeness: 1.000

[PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.056 — The relevant information for answering t

In [19]:
# Step J — Reranker on top of equal-weight hybrid (Exp I / trial-log Experiment I).
# Step I (bm25=0.3) showed weight tuning can't simultaneously improve completeness and adherence.
# Hypothesis: cross-encoder reranker filters BM25 noise *after* retrieval — keeps coverage, cuts irrelevant chunks.
# Config: equal-weight hybrid over-fetches fetch_k=40, cross-encoder reranks to top_n=20.
# Baseline (dense only):     Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
# Step H (hybrid, bm25=0.5): Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%
# Step I (hybrid, bm25=0.3): Rel 0.132 / Util 0.054 / Comp 0.524 / Adh 40%
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
FETCH_K       = 40
TOP_N         = 20

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_j = []

for i, sample in enumerate(samples):
    question   = sample["question"]
    raw_doc    = sample["documents"][0]
    coll_name  = f"cuad_{i}_{COLLECTION_TAG}"

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )

    t0 = time.perf_counter()
    vector_store, cached = get_or_build_vector_store(
        chunks, embeddings,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    elapsed = time.perf_counter() - t0
    print(f"[timing] vector store : {elapsed:.3f}s  ({'cache hit' if cached else f'{len(chunks)} chunks embedded'})")

    retriever = get_hybrid_reranked_retriever(
        vector_store, chunks, reranker, fetch_k=FETCH_K, top_n=TOP_N
    )

    t0 = time.perf_counter()
    retrieved_chunks = retriever.invoke(question)
    print(f"[timing] retrieve+rerank (fetch={FETCH_K}→top-{TOP_N}) : {time.perf_counter() - t0:.3f}s")

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  → {our_response[:100]}...")

    results_j.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

print(f"\nDone — {len(results_j)} samples processed.")


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.290s  (291 chunks embedded)
[timing] retrieve+rerank (fetch=40→top-20) : 0.470s
[timing] retriever: 0.124s
[timing] retriever: 0.001s
[timing] llm      : 55.418s
  → I do not have enough information in the provided context to answer this question. There is no mentio...

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.599s  (47 chunks embedded)
[timing] retrieve+rerank (fetch=40→top-20) : 0.367s
[timing] retriever: 0.140s
[timing] retriever: 0.001s
[timing] llm      : 25.001s
  → The provided text does not contain any information regarding a license granted by one party to its c...

[3/20] The date when the contract is effective ...
[timing] vector store : 1.889s  (168 chunks embedded)
[timing] retrieve+rerank (fetch=40→top-20) : 0.437s
[timing] retriever: 0.135s
[timing] retriever: 0.000s
[timin

In [20]:
# Step J — TRACe evaluation (hybrid equal-weight + cross-encoder reranker)
JUDGE_N_RUNS = 3

print(f"Evaluating Step J results (hybrid+reranker, fetch={FETCH_K}→top-{TOP_N}, n_runs={JUDGE_N_RUNS})...\n")
for r in results_j:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_parsed_llm_response"]            = scores.parsed_llm_response

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

avg_rel  = sum(r["our_relevance"]    for r in results_j) / len(results_j)
avg_util = sum(r["our_utilization"]  for r in results_j) / len(results_j)
avg_comp = sum(r["our_completeness"] for r in results_j) / len(results_j)
adh_rate = sum(r["our_adherence"]    for r in results_j) / len(results_j)

print(f"Step J (hybrid+reranker)    — Rel {avg_rel:.3f} / Util {avg_util:.3f} / Comp {avg_comp:.3f} / Adh {adh_rate:.0%}")
print(f"Step I (hybrid, bm25=0.3)   — Rel 0.132 / Util 0.054 / Comp 0.524 / Adh 40%")
print(f"Step H (hybrid, bm25=0.5)   — Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

Evaluating Step J results (hybrid+reranker, fetch=40→top-20, n_runs=3)...

[FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence in the respo
  Relevance   : 0.123 — The relevant information for answering this question can be found in Section 12c of Docume
  Utilization : 0.045
  Completeness: 0.000

[FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The relevant document for an
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — The response as a whole is supported by the documents. The first sentence in the response,
  Relevance   : 0.035 — The relevant information for answering this question can be found in Section 2a of documen
  Utilization : 0.035


In [23]:
# Step K — HyDE (Hypothetical Document Embeddings)
# Root cause of completeness gap: vocabulary mismatch between query and contract clauses.
# Hypothesis: generating a hypothetical clause and embedding *that* (instead of the raw query)
#   should close the mismatch without introducing BM25 noise.
# Dense-only baseline: Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
# Step J (hybrid+reranker): Rel 0.135 / Util 0.091 / Comp 0.479 / Adh 20%  (worst adherence)
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 20

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_k = []

for i, sample in enumerate(samples):
    question   = sample["question"]
    raw_doc    = sample["documents"][0]
    coll_name  = f"cuad_{i}_{COLLECTION_TAG}"

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )

    t0 = time.perf_counter()
    vector_store, cached = get_or_build_vector_store(
        chunks, embeddings,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    elapsed = time.perf_counter() - t0
    print(f"[timing] vector store : {elapsed:.3f}s  ({'cache hit' if cached else f'{len(chunks)} chunks embedded'})")

    retriever = get_hyde_retriever(vector_store, embeddings, llm, top_k=TOP_K)

    t0 = time.perf_counter()
    retrieved_chunks = retriever.invoke(question)
    print(f"[timing] hyde retrieve (top-{TOP_K}) : {time.perf_counter() - t0:.3f}s")

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  → {our_response[:100]}...")

    results_k.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

print(f"\nDone — {len(results_k)} samples processed.")



[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 3.215s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 22.708s
[timing] llm      : 16.393s
[timing] llm      : 20.862s
  → I do not have enough information in the provided context to determine if one party is required to de...

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.721s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 14.752s
[timing] llm      : 14.677s
[timing] llm      : 22.949s
  → The provided text does not mention whether a license is granted by one party to its counterparty. Ba...

[3/20] The date when the contract is effective ...
[timing] vector store : 2.056s  (168 chunks embedded)
[timing] hyde retrieve (top-20) : 12.783s
[timing] llm      : 18.100s
[timing] llm      : 22.590s
  → The agreement becomes effective when one or more counterparts have been signed by 

In [24]:
# Step K — TRACe evaluation (HyDE)
JUDGE_N_RUNS = 3

print(f"Evaluating Step K results (HyDE, top_k={TOP_K}, n_runs={JUDGE_N_RUNS})...\n")
for r in results_k:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_parsed_llm_response"]            = scores.parsed_llm_response

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

avg_rel  = sum(r["our_relevance"]    for r in results_k) / len(results_k)
avg_util = sum(r["our_utilization"]  for r in results_k) / len(results_k)
avg_comp = sum(r["our_completeness"] for r in results_k) / len(results_k)
adh_rate = sum(r["our_adherence"]    for r in results_k) / len(results_k)

print(f"Step K (HyDE)               — Rel {avg_rel:.3f} / Util {avg_util:.3f} / Comp {avg_comp:.3f} / Adh {adh_rate:.0%}")
print(f"Step J (hybrid+reranker)    — Rel 0.135 / Util 0.091 / Comp 0.479 / Adh 20%")
print(f"Step I (hybrid, bm25=0.3)   — Rel 0.132 / Util 0.054 / Comp 0.524 / Adh 40%")
print(f"Step H (hybrid, bm25=0.5)   — Rel 0.112 / Util 0.086 / Comp 0.590 / Adh 30%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

Evaluating Step K results (HyDE, top_k=20, n_runs=3)...

[FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence of the respo
  Relevance   : 0.040 — The relevant documents for answering this question are those that discuss obligations and 
  Utilization : 0.040
  Completeness: 1.000

[FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence in the respo
  Relevance   : 0.605 — The provided text appears to be a contract or agreement, which may contain information abo
  Utilization : 0.378
  Completeness: 0.625

[PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response,
  Relevance   : 0.409 — The relevant information for answering th

In [10]:
# Step M1 — Purpose-built sentence encoder (BAAI/bge-large-en-v1.5) + HyDE
# Hypothesis A: bge-large is contrastive-trained for cosine retrieval — better geometry than
#   nomic's CLS-vector even though it is general-domain (not legal-specific).
# Config: identical HyDE pipeline as Step K; only the embedder changes.
# Step K (HyDE + nomic):  Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%
# Baseline (dense only):  Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
BGE_MODEL = "BAAI/bge-large-en-v1.5"
embeddings_bge = create_embeddings("huggingface", BGE_MODEL)

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 20

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG_M1 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_bge"

results_m1 = []

for i, sample in enumerate(samples):
    question   = sample["question"]
    raw_doc    = sample["documents"][0]
    coll_name  = f"cuad_{i}_{COLLECTION_TAG_M1}"

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )

    t0 = time.perf_counter()
    vector_store, cached = get_or_build_vector_store(
        chunks, embeddings_bge,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    elapsed = time.perf_counter() - t0
    print(f"[timing] vector store : {elapsed:.3f}s  ({'cache hit' if cached else f'{len(chunks)} chunks embedded'})")

    retriever = get_hyde_retriever(vector_store, embeddings_bge, llm, top_k=TOP_K)

    t0 = time.perf_counter()
    retrieved_chunks = retriever.invoke(question)
    print(f"[timing] hyde retrieve (top-{TOP_K}) : {time.perf_counter() - t0:.3f}s")

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  \u2192 {our_response[:100]}...")

    results_m1.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

print(f"\nDone — {len(results_m1)} samples processed.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.532s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 26.364s
[timing] llm      : 18.700s
[timing] llm      : 24.722s
  → I do not have enough information in the provided context to determine if a party is required to depo...

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.885s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 23.871s
[timing] llm      : 13.441s
[timing] llm      : 20.565s
  → Based on the provided context, there is no mention of a license granted by one party to its counterp...

[3/20] The date when the contract is effective ...
[timing] vector store : 2.648s  (168 chunks embedded)
[timing] hyde retrieve (top-20) : 23.042s
[timing] llm      : 10.314s
[timing] llm      : 28.313s
  → The Agreement becomes effective when one or more counterparts have been signed by 

In [11]:
# Step M1 — TRACe evaluation (bge-large-en-v1.5 + HyDE)
JUDGE_N_RUNS = 3

print(f"Evaluating Step M1 results (bge-large + HyDE, top_k={TOP_K}, n_runs={JUDGE_N_RUNS})...\n")
for r in results_m1:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_parsed_llm_response"]            = scores.parsed_llm_response

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

avg_rel  = sum(r["our_relevance"]    for r in results_m1) / len(results_m1)
avg_util = sum(r["our_utilization"]  for r in results_m1) / len(results_m1)
avg_comp = sum(r["our_completeness"] for r in results_m1) / len(results_m1)
adh_rate = sum(r["our_adherence"]    for r in results_m1) / len(results_m1)

print(f"Step M1 (bge-large + HyDE)  — Rel {avg_rel:.3f} / Util {avg_util:.3f} / Comp {avg_comp:.3f} / Adh {adh_rate:.0%}")
print(f"Step K  (HyDE + nomic)      — Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

Evaluating Step M1 results (bge-large + HyDE, top_k=20, n_runs=3)...

[FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.238 — The relevant information for answering this question can be found in sections 12a, 13a, 14
  Utilization : 0.146
  Completeness: 0.614

[FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence claims that 
  Relevance   : 0.193 — The relevant information for answering this question can be found in PART 1 of the agreeme
  Utilization : 0.129
  Completeness: 0.668

[PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.111 — The relevant document for an

In [12]:
# Step M2 — Legal-domain BERT (nlpaueb/legal-bert-base-uncased) + HyDE
# Hypothesis B: legal-bert was trained on US legal text — its token embeddings sit closer to
#   CUAD contract vocabulary than any general-domain model, even though CLS-pooling is not
#   contrastive-trained.
# Config: identical HyDE pipeline as Step K; only the embedder changes.
# Step M1 (bge-large + HyDE): results above
# Step K  (HyDE + nomic):     Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%
# Baseline (dense only):      Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%
LEGAL_BERT_MODEL = "nlpaueb/legal-bert-base-uncased"
embeddings_legal = create_embeddings("huggingface", LEGAL_BERT_MODEL)

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50
TOP_K         = 20

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG_M2 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_legalbert"

results_m2 = []

for i, sample in enumerate(samples):
    question   = sample["question"]
    raw_doc    = sample["documents"][0]
    coll_name  = f"cuad_{i}_{COLLECTION_TAG_M2}"

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )

    t0 = time.perf_counter()
    vector_store, cached = get_or_build_vector_store(
        chunks, embeddings_legal,
        persist_directory=CHROMA_PERSIST_DIR,
        collection_name=coll_name,
    )
    elapsed = time.perf_counter() - t0
    print(f"[timing] vector store : {elapsed:.3f}s  ({'cache hit' if cached else f'{len(chunks)} chunks embedded'})")

    retriever = get_hyde_retriever(vector_store, embeddings_legal, llm, top_k=TOP_K)

    t0 = time.perf_counter()
    retrieved_chunks = retriever.invoke(question)
    print(f"[timing] hyde retrieve (top-{TOP_K}) : {time.perf_counter() - t0:.3f}s")

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  \u2192 {our_response[:100]}...")

    results_m2.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

print(f"\nDone — {len(results_m2)} samples processed.")

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 2.483s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 24.563s
[timing] llm      : 19.964s
[timing] llm      : 22.281s
  → I do not have enough information to answer this question as the provided context does not mention so...

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.344s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 19.515s
[timing] llm      : 15.864s
[timing] llm      : 35.683s
  → The provided context does not mention any license granted by one party to its counterparty. The agre...

[3/20] The date when the contract is effective ...
[timing] vector store : 1.243s  (168 chunks embedded)
[timing] hyde retrieve (top-20) : 23.527s
[timing] llm      : 14.420s
[timing] llm      : 19.905s
  → The contract is dated as of September 27, 2018....

[4/20] Does the contract conta

In [13]:
# Step M2 — TRACe evaluation (legal-bert-base-uncased + HyDE)
JUDGE_N_RUNS = 3

print(f"Evaluating Step M2 results (legal-bert + HyDE, top_k={TOP_K}, n_runs={JUDGE_N_RUNS})...\n")
for r in results_m2:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_parsed_llm_response"]            = scores.parsed_llm_response

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

avg_rel  = sum(r["our_relevance"]    for r in results_m2) / len(results_m2)
avg_util = sum(r["our_utilization"]  for r in results_m2) / len(results_m2)
avg_comp = sum(r["our_completeness"] for r in results_m2) / len(results_m2)
adh_rate = sum(r["our_adherence"]    for r in results_m2) / len(results_m2)

print(f"Step M2 (legal-bert + HyDE) — Rel {avg_rel:.3f} / Util {avg_util:.3f} / Comp {avg_comp:.3f} / Adh {adh_rate:.0%}")
print(f"Step M1 (bge-large + HyDE)  — Rel <from above>")
print(f"Step K  (HyDE + nomic)      — Rel 0.334 / Util 0.194 / Comp 0.578 / Adh 25%")
print(f"Baseline (dense only)       — Rel 0.173 / Util 0.097 / Comp 0.564 / Adh 55%")

Evaluating Step M2 results (legal-bert + HyDE, top_k=20, n_runs=3)...

[FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The relevant documents for a
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence claims that ther
  Relevance   : 0.104 — The relevant information for answering this question can be found in sections related to m
  Utilization : 0.086
  Completeness: 0.825

[FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The document that contains u
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[FAIL] Does the contract contain a clause that would award either party

In [14]:
# Inspect the top-k retrieved chunks for one sample.
# Set SAMPLE_IDX to 0..N_SAMPLES-1. Sample 1 (Q2) is the most diagnostic — ref_relevance=0.833 but ours=0.
SAMPLE_IDX = 1

r = results[SAMPLE_IDX]
print(f"Q: {r['question']}\n")
print(f"Retrieved {len(r['retrieved_chunks'])} chunks:\n")
for j, chunk in enumerate(r["retrieved_chunks"]):
    print(f"--- chunk {j} ---")
    print(chunk.page_content)
    print()

NameError: name 'results' is not defined

### 4. TRACe Evaluation

We evaluate our generated responses using the TRACe framework (paper: RAGBench, Friel et al.).

| Metric | What | How |
|---|---|---|
| **Adherence** | Is our response fully grounded in context? | LLM-as-judge |
| **Relevance** | Fraction of context relevant to query | From dataset (GPT-4 annotated) |
| **Utilization** | Fraction of context used in response | From dataset (GPT-4 annotated) |
| **Completeness** | Fraction of relevant context captured | From dataset (GPT-4 annotated) |

Relevance/Utilization/Completeness are properties of the document and retrieval setup (fixed per sample),
so we use the dataset's pre-computed GPT-4 labels as reference. We compute **Adherence** ourselves
for our generated response using an LLM judge.

In [10]:
import json

# n_runs > 1 averages the judge over N calls to damp LLM-as-judge variance.
# Pair with a deterministic judge (temperature=0 above) for tightest signal.
JUDGE_N_RUNS = 3

print(f"Evaluating TRACe metrics for each response (n_runs={JUDGE_N_RUNS})...\n")
for r in results:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_parsed_llm_response"]            = scores.parsed_llm_response

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

Evaluating TRACe metrics for each response (n_runs=3)...

[PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents because it does not address the 
  Relevance   : 0.090 — The relevant information for answering this question can be found in Articles III and VII 
  Utilization : 0.050
  Completeness: 0.562

[FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — Claim a: The response states that there is no mention of a license being granted by one pa
  Relevance   : 0.123 — The relevant information for answering this question can be found in document 1, which con
  Utilization : 0.091
  Completeness: 0.736

[PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence of the response 
  Relevance   : 0.045 — The relevant information for answering t

### 5. Results Summary

In [11]:
rows = []
for r in results:
    rows.append({
        "Question"          : r["question"][:50] + "...",
        "Our Adh."          : "✓" if r["our_adherence"] else "✗",
        "Ref Adh."          : "✓" if r["ref_adherence"] else "✗",
        "Our Rel."          : fmt(r["our_relevance"]),
        "Ref Rel."          : fmt(r["ref_relevance"]),
        "Our Util."         : fmt(r["our_utilization"]),
        "Ref Util."         : fmt(r["ref_utilization"]),
        "Our Comp."         : fmt(r["our_completeness"]),
        "Ref Comp."         : fmt(r["ref_completeness"]),
        "RAGAS Faith."      : fmt(r["ragas_faithfulness"]),
        "TruLens Ground."   : fmt(r["trulens_groundedness"]),
    })

df = pd.DataFrame(rows)
print("=" * 100)
print("TRACe Evaluation — CUAD RAGBench")
print("=" * 100)
print(df.to_string(index=False))

our_adh_rate = sum(r["our_adherence"] for r in results) / len(results)
ref_adh_rate = sum(r["ref_adherence"] for r in results) / len(results)

print(f"\nOur Adherence Rate : {our_adh_rate:.0%}  ({sum(r['our_adherence'] for r in results)}/{len(results)} samples)")
print(f"Ref Adherence Rate : {ref_adh_rate:.0%}  ({sum(r['ref_adherence'] for r in results)}/{len(results)} samples)")

print(f"\nAvg Our Relevance   : {sum(r['our_relevance'] for r in results)/len(results):.3f}")
print(f"Avg Our UtilizatiowHln : {sum(r['our_utilization'] for r in results)/len(results):.3f}")
print(f"Avg Our Completeness: {sum(r['our_completeness'] for r in results)/len(results):.3f}")

non_null = [r for r in results if r["ref_relevance"] is not None]
if non_null:
    print(f"\nAvg Ref Relevance   : {sum(r['ref_relevance'] for r in non_null)/len(non_null):.3f}")
    print(f"Avg Ref Utilization : {sum(r['ref_utilization'] for r in non_null)/len(non_null):.3f}")
    print(f"Avg Ref Completeness: {sum(r['ref_completeness'] for r in non_null)/len(non_null):.3f}")

print("\n--- Per-sample responses ---")
for i, r in enumerate(results):
    print(f"\n[{i+1}] Q: {r['question']}")
    print(f"     Our : {r['our_response']}")
    print(f"     Ref : {r['ref_response']}")

TRACe Evaluation — CUAD RAGBench
                                             Question Our Adh. Ref Adh. Our Rel. Ref Rel. Our Util. Ref Util. Our Comp. Ref Comp. RAGAS Faith. TruLens Ground.
Is one party required to deposit its source code i...        ✓        ✓    0.090    0.000     0.050     0.000     0.562     1.000          N/A             N/A
Does the contract contain a license granted by one...        ✗        ✓    0.123    0.833     0.091     0.036     0.736     0.043          N/A             N/A
          The date when the contract is effective ...        ✓        ✓    0.045    0.003     0.045     0.003     1.000     1.000          N/A             N/A
Does the contract contain a clause that would awar...        ✗        ✗    0.169    0.038     0.066     0.003     0.389     0.091          N/A             N/A
                          The date of the contract...        ✗        ✓    0.000    0.004     0.000     0.002     0.000     0.500          N/A             N/A
On what date 